# WarpTracer — fusion and batch benchmark
Choose a GPU runtime for CUDA timings. This compares fused/unfused physics and independent car counts, with 108 LiDAR rays at 60 Hz. Each trial warms up for at least two real seconds. CPU runs also work; start with fewer cars there.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

repo = Path("/content/WarpTracer-rigidbody")
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--single-branch", "--branch",
                    "performance-prototype", "https://github.com/yuywe/WarpTracer.git", str(repo)], check=True)
os.chdir(repo)
subprocess.run(["git", "fetch", "origin", "performance-prototype:refs/remotes/origin/performance-prototype"], check=True)
if subprocess.run(["git", "show-ref", "--verify", "--quiet", "refs/heads/performance-prototype"]).returncode:
    subprocess.run(["git", "branch", "--no-track", "performance-prototype", "origin/performance-prototype"], check=True)
subprocess.run(["git", "switch", "performance-prototype"], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", "performance-prototype"], check=True)
os.environ["UV_PYTHON"] = sys.executable
subprocess.run(["uv", "sync", "--locked", "--extra", "viz"], check=True)


In [ ]:
physics = "lean"
envs = [1, 64, 256, 1024]  # use [1] with Newton
integrator = "fused"  # "both" compares fusion; use "unfused" with Newton
subprocess.run(["uv", "run", "--locked", "python", "-m", "warptracer.benchmark",
                "--physics", physics, "--backend", "graph", "--integrator", integrator,
                "--envs", *map(str, envs), "--cases", "physics", "lidar", "--seconds", "100",
                "--output", f"outputs/{physics}-benchmark.json"], check=True)


In [ ]:
import json
report = json.loads(Path(f"outputs/{physics}-benchmark.json").read_text())
print(report["device_name"], report["physics_engine"])
for result in report["results"]:
    s = result["summary"]
    print("cars:", result["environments"], result["case"], result["integrator"], result["backend"],
          round(s["median_aggregate_environment_transitions_per_second"]), "aggregate transitions/s;",
          round(s["median_batch_transitions_per_second"]), "batch transitions/s;",
          "variable timing" if s["timing_variable"] else "max/min <= 1.2")
print("Full report:", Path(f"outputs/{physics}-benchmark.json").resolve())
